# Agente Reactivo Simple: Hogar Inteligente

---

### Características de este agente

- **Sensores**: temperatura, humedad, movimiento, luz, humo y puerta
- **Actuadores**: aire acondicionado, calefacción, luces, alarma, aspersores, ventilador y cerradura
- **10 reglas** que cubren emergencias, confort y ahorro de energía

### Algoritmo

```
funcion agente_reactivo(percepcion) return accion
    actualizar_estado(percepcion)
    para cada regla en reglas:
        si regla.condicion(estado) entonces
            return regla.accion(estado)
    return accion_por_defecto
```

---

## 1. Importaciones

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Callable

print("Bibliotecas importadas correctamente")

---

## 2. Estado del Hogar

El agente mantiene un estado interno con los valores de los sensores y el estado de los actuadores.

In [ ]:
class EstadoHogar:
    """Estado interno del hogar inteligente."""

    def __init__(self):
        # Sensores
        self.sensores: Dict[str, any] = {
            "temperatura": 22.0,   # °C
            "humedad": 50.0,      # %
            "movimiento": False,  # True/False
            "luz": 500.0,        # lux
            "humo": False,        # True/False
            "puerta_abierta": False,  # True/False
        }

        # Actuadores
        self.actuadores: Dict[str, bool] = {
            "aire_acondicionado": False,
            "calefaccion": False,
            "luces": False,
            "alarma": False,
            "aspersores": False,
            "ventilador": False,
            "cerradura": True,  # True = bloqueada
        }

    def actualizar_sensor(self, nombre: str, valor):
        """Actualiza el valor de un sensor."""
        if nombre in self.sensores:
            self.sensores[nombre] = valor

    def activar_actuador(self, nombre: str, estado: bool):
        """Cambia el estado de un actuador."""
        if nombre in self.actuadores:
            self.actuadores[nombre] = estado

    def resumen(self) -> str:
        """Retorna un resumen legible del estado actual."""
        lineas = [
            f"Temperatura: {self.sensores['temperatura']:.1f}°C | "
            f"Humedad: {self.sensores['humedad']:.0f}% | "
            f"Luz: {self.sensores['luz']:.0f} lux",
            f"Movimiento: {'Si' if self.sensores['movimiento'] else 'No'} | "
            f"Humo: {'DETECTADO' if self.sensores['humo'] else 'No'} | "
            f"Puerta: {'Abierta' if self.sensores['puerta_abierta'] else 'Cerrada'}",
            f"A/C: {'ON' if self.actuadores['aire_acondicionado'] else 'OFF'} | "
            f"Calefaccion: {'ON' if self.actuadores['calefaccion'] else 'OFF'} | "
            f"Luces: {'ON' if self.actuadores['luces'] else 'OFF'} | "
            f"Alarma: {'ON' if self.actuadores['alarma'] else 'OFF'} | "
            f"Aspersores: {'ON' if self.actuadores['aspersores'] else 'OFF'} | "
            f"Ventilador: {'ON' if self.actuadores['ventilador'] else 'OFF'} | "
            f"Cerradura: {'Bloqueada' if self.actuadores['cerradura'] else 'Liberada'}",
        ]
        return "\n".join(lineas)


# Instancia inicial
estado = EstadoHogar()
print("Estado del hogar inicializado:")
print(estado.resumen())

---

## 3. Definición de Reglas



Las reglas están ordenadas de mayor a menor prioridad: las emergencias se evalúan primero, luego seguridad, confort y ahorro.

In [ ]:
@dataclass
class Regla:
    """Representa una regla si-entonces del agente reactivo."""
    nombre: str
    condicion: Callable[[EstadoHogar], bool]
    accion: Callable[[EstadoHogar], str]
    descripcion: str

    def evaluar(self, estado: EstadoHogar) -> bool:
        """Verifica si la condición se cumple."""
        return self.condicion(estado)

    def ejecutar(self, estado: EstadoHogar) -> str:
        """Ejecuta la acción y retorna su descripción."""
        return self.accion(estado)


print("Clase Regla definida")

---

## 4. Las 10 Acciones del Agente

El agente puede realizar **10 acciones** distintas, organizadas por categoría:

| # | Categoría | Acción | Condición (SI) | Respuesta (ENTONCES) |
|---|-----------|--------|----------------|---------------------|
| 1 | Emergencia | Detectar incendio | Se detecta humo | Alarma + aspersores |
| 2 | Emergencia | Recuperar de incendio | Humo desaparece y alarma activa | Apagar alarma y aspersores |
| 3 | Seguridad | Asegurar puerta abierta | Puerta abierta sin movimiento | Bloquear cerradura |
| 4 | Confort | Activar A/C | Temperatura > 28°C | Encender A/C |
| 5 | Confort | Activar calefacción | Temperatura < 16°C | Encender calefacción |
| 6 | Confort | Apagar climatización | Temperatura entre 20-25°C y HVAC encendido | Apagar A/C y calefacción |
| 7 | Confort | Ventilar por humedad | Humedad > 75% | Encender ventilador |
| 8 | Confort | Apagar ventilador | Humedad < 40% y ventilador encendido | Apagar ventilador |
| 9 | Conveniencia | Encender luces | Poca luz y hay movimiento | Encender luces |
| 10 | Ahorro | Apagar luces | Luz natural suficiente y luces encendidas | Apagar luces |

In [ ]:
# ============================================================
# EMERGENCIAS (se evalúan primero)
# ============================================================

# --- Accion 1: Detectar incendio ---
regla_incendio = Regla(
    nombre="Deteccion de Incendio",
    condicion=lambda e: e.sensores["humo"] == True,
    accion=lambda e: (
        e.activar_actuador("alarma", True),
        e.activar_actuador("aspersores", True),
        "INCENDIO detectado! Alarma y aspersores activados."
    )[-1],
    descripcion="Si hay humo, activar alarma y aspersores"
)

# --- Accion 2: Recuperar de incendio ---
regla_fin_incendio = Regla(
    nombre="Fin de Incendio",
    condicion=lambda e: e.sensores["humo"] == False and e.actuadores["alarma"] == True,
    accion=lambda e: (
        e.activar_actuador("alarma", False),
        e.activar_actuador("aspersores", False),
        "Incendio controlado. Alarma y aspersores desactivados."
    )[-1],
    descripcion="Si no hay humo y alarma activa, desactivar emergencia"
)

# ============================================================
# SEGURIDAD
# ============================================================

# --- Accion 3: Asegurar puerta abierta sin nadie ---
regla_puerta_insegura = Regla(
    nombre="Puerta Abierta sin Supervision",
    condicion=lambda e: e.sensores["puerta_abierta"] == True and e.sensores["movimiento"] == False,
    accion=lambda e: (
        e.activar_actuador("cerradura", True),
        "Puerta abierta sin personas presentes. Cerradura bloqueada por seguridad."
    )[-1],
    descripcion="Si puerta abierta y no hay movimiento, bloquear cerradura"
)

# ============================================================
# CONFORT - Clima
# ============================================================

# --- Accion 4: Activar A/C por calor ---
regla_calor = Regla(
    nombre="Activar A/C por Calor",
    condicion=lambda e: e.sensores["temperatura"] > 28.0 and not e.actuadores["aire_acondicionado"],
    accion=lambda e: (
        e.activar_actuador("aire_acondicionado", True),
        e.activar_actuador("calefaccion", False),
        f"Hace calor ({e.sensores['temperatura']:.1f}°C). Aire acondicionado activado."
    )[-1],
    descripcion="Si temperatura > 28°C, activar aire acondicionado"
)

# --- Accion 5: Activar calefaccion por frio ---
regla_frio = Regla(
    nombre="Activar Calefaccion por Frio",
    condicion=lambda e: e.sensores["temperatura"] < 16.0 and not e.actuadores["calefaccion"],
    accion=lambda e: (
        e.activar_actuador("calefaccion", True),
        e.activar_actuador("aire_acondicionado", False),
        f"Hace frio ({e.sensores['temperatura']:.1f}°C). Calefaccion activada."
    )[-1],
    descripcion="Si temperatura < 16°C, activar calefaccion"
)

# --- Accion 6: Apagar climatizacion con temperatura ideal ---
regla_temperatura_ideal = Regla(
    nombre="Temperatura Ideal - Apagar Climatizacion",
    condicion=lambda e: 20.0 <= e.sensores["temperatura"] <= 25.0 and (
        e.actuadores["aire_acondicionado"] or e.actuadores["calefaccion"]
    ),
    accion=lambda e: (
        e.activar_actuador("aire_acondicionado", False),
        e.activar_actuador("calefaccion", False),
        f"Temperatura ideal ({e.sensores['temperatura']:.1f}°C). A/C y calefaccion apagados."
    )[-1],
    descripcion="Si temperatura entre 20-25°C y HVAC encendido, apagarlo"
)

# --- Accion 7: Ventilar por humedad alta ---
regla_humedad_alta = Regla(
    nombre="Humedad Alta - Activar Ventilacion",
    condicion=lambda e: e.sensores["humedad"] > 75.0 and not e.actuadores["ventilador"],
    accion=lambda e: (
        e.activar_actuador("ventilador", True),
        f"Humedad alta ({e.sensores['humedad']:.0f}%). Ventilador activado."
    )[-1],
    descripcion="Si humedad > 75%, activar ventilador"
)

# --- Accion 8: Apagar ventilador con humedad baja ---
regla_humedad_baja = Regla(
    nombre="Humedad Baja - Apagar Ventilador",
    condicion=lambda e: e.sensores["humedad"] < 40.0 and e.actuadores["ventilador"],
    accion=lambda e: (
        e.activar_actuador("ventilador", False),
        f"Humedad baja ({e.sensores['humedad']:.0f}%). Ventilador apagado."
    )[-1],
    descripcion="Si humedad < 40% y ventilador encendido, apagarlo"
)

# ============================================================
# CONVENIENCIA Y AHORRO
# ============================================================

# --- Accion 9: Encender luces con poca luz ---
regla_luz_baja = Regla(
    nombre="Encender Luces por Poca Luz",
    condicion=lambda e: e.sensores["luz"] < 200 and e.sensores["movimiento"] and not e.actuadores["luces"],
    accion=lambda e: (
        e.activar_actuador("luces", True),
        f"Poca luz ({e.sensores['luz']:.0f} lux) con persona presente. Luces encendidas."
    )[-1],
    descripcion="Si luz < 200 lux y hay movimiento, encender luces"
)

# --- Accion 10: Apagar luces con luz natural suficiente ---
regla_luz_alta = Regla(
    nombre="Luz Natural Suficiente - Apagar Luces",
    condicion=lambda e: e.sensores["luz"] > 600 and e.actuadores["luces"],
    accion=lambda e: (
        e.activar_actuador("luces", False),
        f"Luz natural suficiente ({e.sensores['luz']:.0f} lux). Luces apagadas por ahorro."
    )[-1],
    descripcion="Si luz > 600 lux y luces encendidas, apagarlas"
)


# ============================================================
# Lista de reglas (ordenadas por prioridad)
# ============================================================
reglas = [
    # Emergencias
    regla_incendio,
    regla_fin_incendio,
    # Seguridad
    regla_puerta_insegura,
    # Confort - Clima
    regla_calor,
    regla_frio,
    regla_temperatura_ideal,
    regla_humedad_alta,
    regla_humedad_baja,
    # Conveniencia y Ahorro
    regla_luz_baja,
    regla_luz_alta,
]

print(f"{len(reglas)} reglas definidas:")
print()
categorias = {
    "Emergencia": [1, 2],
    "Seguridad": [3],
    "Confort": [4, 5, 6, 7, 8],
    "Conveniencia/Ahorro": [9, 10],
}
for cat, nums in categorias.items():
    print(f"  {cat}:")
    for n in nums:
        r = reglas[n-1]
        print(f"    {n}. {r.nombre}: {r.descripcion}")

---

## 5. Agente Reactivo



In [ ]:
class AgenteReactivoHogar:
    """Agente reactivo simple para el control de un hogar inteligente."""

    def __init__(self, estado: EstadoHogar, reglas: List[Regla]):
        self.estado = estado
        self.reglas = reglas
        self.historial: List[Dict] = []

    def percibir(self, percepcion: Dict[str, any]):
        """Actualiza el estado con las nuevas percepciones de los sensores."""
        for sensor, valor in percepcion.items():
            self.estado.actualizar_sensor(sensor, valor)

    def actuar(self, percepcion: Dict[str, any]) -> str:
        """Ciclo completo: percibir, evaluar reglas y ejecutar la primera que coincida."""
        # 1. Percibir
        self.percibir(percepcion)

        # 2. Evaluar reglas en orden y ejecutar la primera que se active
        for regla in self.reglas:
            if regla.evaluar(self.estado):
                resultado = regla.ejecutar(self.estado)
                self.historial.append({"regla": regla.nombre, "accion": resultado})
                return resultado

        # 3. Si ninguna regla se activa
        resultado = "Sin accion: Todo en orden."
        self.historial.append({"regla": "Ninguna", "accion": resultado})
        return resultado

    def mostrar_estado(self):
        """Muestra el estado actual del hogar."""
        print("=" * 55)
        print(self.estado.resumen())
        print("=" * 55)


# Crear instancia del agente
estado = EstadoHogar()
agente = AgenteReactivoHogar(estado, reglas)

print(f"Agente reactivo creado con {len(reglas)} acciones")
agente.mostrar_estado()

---

## 6. Pruebas del Agente

Probamos las 10 acciones del agente con escenarios diferentes. Cada prueba establece todos los sensores para evitar efectos secundarios entre pruebas.

In [ ]:
# Percepcion base: estado neutro para resetear entre pruebas
base = {"temperatura": 22.0, "humedad": 50.0, "movimiento": False, "luz": 500.0, "humo": False, "puerta_abierta": False}

print("=" * 60)
print("PRUEBAS DEL AGENTE REACTIVO")
print("=" * 60)

# --- Prueba 1: Incendio ---
print("\nPRUEBA 1: Deteccion de incendio")
resultado = agente.actuar({**base, "humo": True, "temperatura": 35.0})
print(f"  Resultado: {resultado}")

# --- Prueba 2: Fin de incendio ---
print("\nPRUEBA 2: Fin de incendio (humo desaparece)")
resultado = agente.actuar({**base, "humo": False})
print(f"  Resultado: {resultado}")

# --- Prueba 3: Puerta abierta sin nadie ---
print("\nPRUEBA 3: Puerta abierta sin supervision")
resultado = agente.actuar({**base, "puerta_abierta": True, "movimiento": False})
print(f"  Resultado: {resultado}")

# --- Prueba 4: Calor ---
print("\nPRUEBA 4: Escenario de calor")
resultado = agente.actuar({**base, "temperatura": 32.0})
print(f"  Resultado: {resultado}")

# --- Prueba 5: Frio ---
print("\nPRUEBA 5: Escenario de frio")
resultado = agente.actuar({**base, "temperatura": 10.0})
print(f"  Resultado: {resultado}")

# --- Prueba 6: Temperatura ideal (calefaccion encendida) ---
print("\nPRUEBA 6: Temperatura ideal con HVAC encendido")
agente.estado.activar_actuador("calefaccion", True)
resultado = agente.actuar({**base, "temperatura": 23.0})
print(f"  Resultado: {resultado}")

# --- Prueba 7: Humedad alta ---
print("\nPRUEBA 7: Humedad alta")
resultado = agente.actuar({**base, "humedad": 85.0})
print(f"  Resultado: {resultado}")

# --- Prueba 8: Humedad baja (ventilador encendido) ---
print("\nPRUEBA 8: Humedad baja con ventilador encendido")
resultado = agente.actuar({**base, "humedad": 30.0})
print(f"  Resultado: {resultado}")

# --- Prueba 9: Poca luz con movimiento ---
print("\nPRUEBA 9: Poca luz con persona presente")
resultado = agente.actuar({**base, "luz": 100.0, "movimiento": True})
print(f"  Resultado: {resultado}")

# --- Prueba 10: Luz natural suficiente (luces encendidas) ---
print("\nPRUEBA 10: Luz natural suficiente con luces encendidas")
agente.estado.activar_actuador("luces", True)
resultado = agente.actuar({**base, "luz": 700.0})
print(f"  Resultado: {resultado}")

# --- Estado final e historial ---
print("\n" + "=" * 60)
print("ESTADO FINAL:")
agente.mostrar_estado()

print("\nHISTORIAL DE ACCIONES:")
for i, h in enumerate(agente.historial, 1):
    print(f"  {i}. [{h['regla']}] {h['accion']}")